# [PyBroMo](http://opensmfs.github.io/PyBroMo/) - B.3 ALEX simulation - Generate photon timestamps

<small><i>
This notebook is part of <a href="http://opensmfs.github.io/PyBroMo" target="_blank">PyBroMo</a> a 
python-based single-molecule Brownian motion diffusion simulator 
that simulates confocal smFRET
experiments.
</i></small>

## *Overview*

*In this notebook we show how to generate timestamps for Alternating Laser Excitation (ALEX) simulations from saved diffusion traces*.

In [ ]:
%matplotlib inline
import numpy as np
import tables
import matplotlib.pyplot as plt
import seaborn as sns
import pybromo as pbm
print('Numpy version:', np.__version__)
print('PyTables version:', tables.__version__)
print('PyBroMo version:', pbm.__version__)

## 1. Simulation setup

First, we define the simulation parameters and generate trajectories if they don't exist yet.

In [ ]:
# Simulation parameters
t_step = 10e-6       # Simulation time step (seconds)
t_max = 0.5          # Time duration of the simulation (seconds)
Du = 100.0           # Diffusion coefficient (um^2 / s)
D = Du*(1e-6)**2     # m^2 / s

# Simulation box definition
box = pbm.Box(x1=-2.e-6, x2=2.e-6, y1=-2.e-6, y2=2.e-6, z1=-3e-6, z2=3e-6)

# Particles definition (e.g., 2 populations with 2 particles each)
rs = np.random.RandomState(seed=1)
P = pbm.Particles.from_specs(
    num_particles=(2, 2),
    D=(D, D),
    box=box, rs=rs)

# PSF definition
psf = pbm.psflib.GaussianPSF()

S = pbm.ParticlesSimulation(t_step=t_step, t_max=t_max, particles=P, box=box, psf=psf)

# Run trajectory simulation
# Note: total_emission=False is required for simulate_timestamps_alex to work correctly
S.simulate_diffusion(total_emission=False)

## 2. ALEX Timestamps Simulation

Now we define the ALEX parameters and generate the timestamps.

In [ ]:
# ALEX parameters
alex_period = 100e-6  # ALEX period in seconds (10 kHz)
d_duty = 0.4          # Donor laser duty cycle
a_duty = 0.4          # Acceptor laser duty cycle

# Optical and sample parameters for each population
populations = [slice(0, 2), slice(2, 4)]
max_rates_d = [200e3, 200e3]  # Peak emission rates for D-laser (cps)
max_rates_a = [200e3, 200e3]  # Peak emission rates for A-laser (cps)
E_values = [0.1, 0.7]         # FRET efficiencies

leakage = 0.1                 # D emission leakage into A channel
direct_exc = 0.05             # Direct excitation of A by D-laser
bg_rate_d = 500               # Background rate D (cps)
bg_rate_a = 500               # Background rate A (cps)

# Generate ALEX timestamps
S.simulate_timestamps_alex(
    populations=populations,
    max_rates_d_laser=max_rates_d,
    max_rates_a_laser=max_rates_a,
    E_values=E_values,
    leakage=leakage,
    direct_exc=direct_exc,
    bg_rate_d=bg_rate_d,
    bg_rate_a=bg_rate_a,
    alex_period=alex_period,
    d_duty=d_duty,
    a_duty=a_duty,
    overwrite=True
)

## 3. Results Overview

We can check how many timestamps were generated for each channel.

In [ ]:
print(f"Generated {len(S._timestamps_d)} donor timestamps.")
print(f"Generated {len(S._timestamps_a)} acceptor timestamps.")

The timestamps are stored in the HDF5 file and can be accessed via `S._timestamps_d` and `S._timestamps_a`.